## 6.5 RAG를 위한 에이전트 만들기

### 6.5.2 [실습] 벡터 데이터베이스의 이해와 사용하기

In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

file_path = "datasets/한글맞춤법 표준어규정 해설.pdf"

loader = PyPDFLoader(file_path)
pages = []

async for page in loader.alazy_load():
    pages.append(page)

In [2]:
print("페이지 수:", len(pages))
print("첫번째 페이지 정보: ", pages[0])

페이지 수: 264
첫번째 페이지 정보:  page_content='국립국어원 2018-01-08
발간 등록 번호
11-1371028-000712-01
한글 맞춤법
표준어 규정
해설' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2018-12-20T09:36:29+09:00', 'author': 'admin', 'moddate': '2018-12-20T09:38:09+09:00', 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'total_pages': 264, 'page': 0, 'page_label': '1'}


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(pages)

In [4]:
print(f"총 {len(docs)}개의 청크 생성 완료")
print("각 청크의 길이:", [len(i.page_content) for i in docs])

for i in docs:
    print("[메타데이터]", i.metadata)
    print("[내용]", i.page_content)
    print("=" * 100)

총 510개의 청크 생성 완료
각 청크의 길이: [63, 467, 367, 472, 96, 473, 110, 343, 458, 471, 499, 129, 484, 456, 461, 124, 11, 420, 448, 448, 466, 463, 220, 489, 486, 484, 466, 54, 454, 476, 288, 468, 167, 468, 475, 453, 79, 493, 218, 474, 494, 268, 476, 332, 480, 242, 450, 452, 450, 202, 497, 259, 486, 482, 119, 492, 124, 454, 429, 456, 415, 492, 107, 479, 250, 489, 462, 102, 489, 387, 486, 357, 459, 432, 481, 254, 308, 345, 490, 304, 362, 494, 154, 476, 307, 474, 475, 50, 465, 195, 406, 460, 78, 492, 499, 53, 493, 213, 489, 254, 476, 144, 430, 498, 191, 453, 163, 500, 62, 478, 417, 473, 101, 497, 68, 487, 182, 477, 338, 490, 92, 470, 491, 101, 498, 48, 486, 490, 105, 461, 411, 470, 239, 485, 462, 223, 487, 190, 459, 469, 134, 462, 244, 457, 220, 478, 405, 485, 191, 464, 315, 482, 186, 487, 394, 483, 499, 486, 181, 484, 480, 481, 143, 460, 464, 190, 494, 138, 472, 475, 497, 118, 419, 423, 487, 318, 488, 19, 499, 91, 495, 125, 497, 210, 477, 90, 498, 401, 489, 67, 496, 135, 454, 140, 407, 381, 499, 353

In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

DB_PATH = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=DB_PATH,
    collection_name="korean_pdf"
)

In [6]:
vectorstore.similarity_search("구개음화", k=3)

[Document(id='d4c93b78-6666-4fa1-a939-7f14f559249f', metadata={'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'page_label': '24', 'total_pages': 264, 'creationdate': '2018-12-20T09:36:29+09:00', 'moddate': '2018-12-20T09:38:09+09:00', 'author': 'admin', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'page': 23}, page_content='는 부사 파생 접미사가 결합하여 부사가 되었으므로 구개음화가 실현되었지만, ‘곧이\n어’는 형식 형태소가 아닌 실질 형태소 부사가 결합한 말이므로 구개음화가 실현되지 \n않는다. \n곧이[고지]: 곧-(어근)+-이(부사 파생 접미사)\n곧이어[고디어]: 곧(부사)+이어(부사)\n현재 표준어에서 구개음화는 형태소와 형태소가 결합할 때 일어나는 현상이다. 그러\n므로 ‘마디, 견디다’와 같이 하나의 형태소 내부에서는 구개음화가 일어나지 않는다.'),
 Document(id='1dae9d94-9fbf-479c-b686-9d7efd8438b4', metadata={'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page_label': '245', 'total_pages': 264, 'moddate': '2018-12-20T09:38:09+09:00', 'page': 244, 'creationdate': '2018-12-20T09:36:2